## **1D CNN Model**

In [1]:
import json
import mne
import os
import time
import warnings
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import mne
import numpy as np
import tensorflow as tf

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
RUNS_DIR = OUTPUTS_DIR / 'runs'
LATEST_DIR = OUTPUTS_DIR / 'latest'

for p in [OUTPUTS_DIR, RUNS_DIR, LATEST_DIR]:
    p.mkdir(parents=True, exist_ok=True)

if not DATA_DIR.exists():
    raise FileNotFoundError(f'Data folder not found: {DATA_DIR}')

LABEL_MAP = {
    'Sleep stage W': 0,
    'Sleep stage 1': 1,
    'Sleep stage 2': 2,
    'Sleep stage 3': 3,
    'Sleep stage 4': 3,
    'Sleep stage R': 4,
}

STAGE_NAMES = {0: 'Wake', 1: 'N1', 2: 'N2', 3: 'N3', 4: 'REM'}
N_CLASSES = 5
EPOCH_SEC = 30
CHANNEL_NAME = 'EEG Fpz-Cz'

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
tf.random.set_seed(42)
np.random.seed(42)

print(f'Project root: {PROJECT_ROOT}')
print(f'Data dir:      {DATA_DIR}')
print(f'TensorFlow:    {tf.__version__}')


Project root: c:\Users\cbamm\1 Jupyter\ECGR 4116\EEG Final Project
Data dir:      c:\Users\cbamm\1 Jupyter\ECGR 4116\EEG Final Project\data
TensorFlow:    2.20.0


In [2]:
import sys
print(sys.executable)
import tensorflow as tf
print(tf.__version__)

c:\Users\cbamm\AppData\Local\Programs\Python\Python310\python.exe
2.20.0


In [ ]:
def extract_raw_epochs(rec_path, hyp_path, channel_name=CHANNEL_NAME, epoch_sec=EPOCH_SEC):
    """Load one subject and return raw 30-second EEG epochs plus stage labels."""
    warnings.filterwarnings('ignore', category=RuntimeWarning)

    raw = mne.io.read_raw_edf(str(rec_path), preload=True, verbose=0)
    annotations = mne.read_annotations(str(hyp_path))

    raw.pick([channel_name])
    data, _ = raw[:]
    sfreq = int(raw.info['sfreq'])
    n_samples = epoch_sec * sfreq

    epochs = []
    labels = []

    for ann in annotations:
        stage = ann['description']
        if stage not in LABEL_MAP:
            continue

        onset = ann['onset']
        duration = ann['duration']
        n_epochs_in_ann = int(duration // epoch_sec)

        for i in range(n_epochs_in_ann):
            start = int((onset + i * epoch_sec) * sfreq)
            end = start + n_samples

            if end > data.shape[1]:
                continue

            epoch = data[0, start:end].astype(np.float32)
            if not np.isfinite(epoch).all():
                continue

            epochs.append(epoch)
            labels.append(LABEL_MAP[stage])

    return np.array(epochs, dtype=np.float32), np.array(labels, dtype=np.int64), sfreq


def load_all_subjects_raw(data_dir=DATA_DIR):
   
    recordings = sorted(data_dir.glob('SC*E0-PSG.edf'))
    if not recordings:
        raise FileNotFoundError(f'No EDF recordings found in {data_dir}')

    all_epochs = []
    all_labels = []
    all_subject_ids = []
    sfreq_seen = None

    print(f'Found {len(recordings)} subject(s):')
    for rec_path in recordings:
        print(f"  - {rec_path.name.split('E0')[0]}")

    for idx, rec_path in enumerate(recordings, 1):
        subject_id = rec_path.name.split('E0')[0]
        hypnogram_files = list(data_dir.glob(f'{subject_id}*-Hypnogram.edf'))
        if not hypnogram_files:
            print(f'[{idx}/{len(recordings)}] {subject_id}: skipped (no hypnogram)')
            continue

        X_subj, y_subj, sfreq = extract_raw_epochs(rec_path, hypnogram_files[0])
        if sfreq_seen is None:
            sfreq_seen = sfreq
        elif sfreq != sfreq_seen:
            raise ValueError(f'Sampling frequency mismatch: expected {sfreq_seen}, got {sfreq}')

        all_epochs.append(X_subj)
        all_labels.append(y_subj)
        all_subject_ids.extend([subject_id] * len(y_subj))

        counts = {cls: int((y_subj == cls).sum()) for cls in range(N_CLASSES)}
        print(
            f'[{idx}/{len(recordings)}] {subject_id}: {len(y_subj)} epochs | '
            f'Wake {counts[0]}  N1 {counts[1]}  N2 {counts[2]}  N3 {counts[3]}  REM {counts[4]}'
        )

    X_all = np.vstack(all_epochs)
    y_all = np.concatenate(all_labels)
    subject_ids = np.array(all_subject_ids)

    return X_all, y_all, subject_ids, sfreq_seen


X_raw, y_all, subject_ids, sfreq = load_all_subjects_raw()

print('\nCombined raw dataset:')
print(f'  X_raw shape:     {X_raw.shape}')
print(f'  y_all shape:     {y_all.shape}')
print(f'  subject_ids:     {subject_ids.shape}')
print(f'  sampling rate:   {sfreq} Hz')
print(f'  epoch duration:  {EPOCH_SEC} sec')
print(f'  samples/epoch:   {X_raw.shape[1]}')

Found 40 subject(s):
  - SC4001
  - SC4002
  - SC4011
  - SC4012
  - SC4021
  - SC4022
  - SC4031
  - SC4032
  - SC4041
  - SC4042
  - SC4051
  - SC4052
  - SC4061
  - SC4062
  - SC4071
  - SC4072
  - SC4081
  - SC4082
  - SC4091
  - SC4092
  - SC4101
  - SC4102
  - SC4111
  - SC4112
  - SC4121
  - SC4122
  - SC4131
  - SC4141
  - SC4142
  - SC4151
  - SC4152
  - SC4161
  - SC4162
  - SC4171
  - SC4172
  - SC4181
  - SC4182
  - SC4191
  - SC4192
  - SC4201
[1/40] SC4001: 2650 epochs | Wake 1997  N1 58  N2 250  N3 220  REM 125
[2/40] SC4002: 2829 epochs | Wake 1885  N1 59  N2 373  N3 297  REM 215
[3/40] SC4011: 2802 epochs | Wake 1856  N1 109  N2 562  N3 105  REM 170
[4/40] SC4012: 2848 epochs | Wake 1824  N1 92  N2 660  N3 96  REM 176
[5/40] SC4021: 2804 epochs | Wake 1907  N1 94  N2 545  N3 95  REM 163
[6/40] SC4022: 2755 epochs | Wake 1871  N1 184  N2 402  N3 119  REM 179
[7/40] SC4031: 2820 epochs | Wake 2008  N1 61  N2 485  N3 57  REM 209
[8/40] SC4032: 2732 epochs | Wake 1957  N1 

In [ ]:
# CNN preprocessing
epoch_mean = X_raw.mean(axis=1, keepdims=True)
epoch_std = X_raw.std(axis=1, keepdims=True) + 1e-8
X_cnn = ((X_raw - epoch_mean) / epoch_std)[..., np.newaxis]
# downcast to float32 to reduce RAM (in-place when possible)
X_cnn = X_cnn.astype(np.float32, copy=False)

print('CNN-ready tensors:')
print(f'  X_cnn shape: {X_cnn.shape}')
print(f'  y_all shape: {y_all.shape}')
print(f'  Example input shape per epoch: {X_cnn[0].shape}')

print('\nClass distribution:')
for cls in range(N_CLASSES):
    count = int((y_all == cls).sum())
    pct = 100 * count / len(y_all)
    print(f"  {STAGE_NAMES[cls]:4s}: {count:5d} ({pct:5.1f}%)")

assert X_raw.ndim == 2, 'Raw epochs should be (n_epochs, n_samples)'
assert X_cnn.ndim == 3, 'CNN input should be (n_epochs, n_samples, channels)'
assert X_cnn.shape[1] == 3000, 'Expected 3000 samples per 30-second epoch at 100 Hz'
assert set(np.unique(y_all)) == {0, 1, 2, 3, 4}, 'Labels should cover the 5 sleep stages'
assert np.isfinite(X_cnn).all(), 'CNN inputs contain NaN/Inf values'

print('\nPreprocessing checks passed. Ready for CNN training.')

CNN-ready tensors:
  X_cnn shape: (109217, 3000, 1)
  y_all shape: (109217,)
  Example input shape per epoch: (3000, 1)

Class distribution:
  Wake: 74435 ( 68.2%)
  N1  :  2843 (  2.6%)
  N2  : 18338 ( 16.8%)
  N3  :  5707 (  5.2%)
  REM :  7894 (  7.2%)

Preprocessing checks passed. Ready for CNN training.


In [15]:
"""
# Balanced data generator for class imbalance (placed before training)
from tensorflow.keras.utils import Sequence
import math

class BalancedSequence(Sequence):
    def __init__(self, X, y, batch_size=64, n_classes=5, rng=None):
        self.X = X
        self.y = y
        self.batch_size = batch_size
        self.n_classes = n_classes
        self.rng = np.random.default_rng() if rng is None else rng

        counts = np.bincount(y, minlength=n_classes).astype(float)
        counts[counts == 0] = 1.0
        inv = 1.0 / counts
        self.class_probs = inv / inv.sum()
        self.indices_by_class = {c: np.where(y == c)[0] for c in range(n_classes)}

    def __len__(self):
        return math.ceil(len(self.y) / self.batch_size)

    def __getitem__(self, idx):
        chosen_classes = self.rng.choice(self.n_classes, size=self.batch_size, p=self.class_probs)
        batch_idx = []
        for c in chosen_classes:
            candidates = self.indices_by_class.get(int(c))
            if len(candidates) == 0:
                batch_idx.append(self.rng.integers(0, len(self.y)))
            else:
                batch_idx.append(int(self.rng.choice(candidates)))
        batch_idx = np.array(batch_idx, dtype=np.int64)
        Xb = self.X[batch_idx]
        yb = self.y[batch_idx]
        return Xb, yb

    def on_epoch_end(self):
        pass
"""

'\n# Balanced data generator for class imbalance (placed before training)\nfrom tensorflow.keras.utils import Sequence\nimport math\n\nclass BalancedSequence(Sequence):\n    def __init__(self, X, y, batch_size=64, n_classes=5, rng=None):\n        self.X = X\n        self.y = y\n        self.batch_size = batch_size\n        self.n_classes = n_classes\n        self.rng = np.random.default_rng() if rng is None else rng\n\n        counts = np.bincount(y, minlength=n_classes).astype(float)\n        counts[counts == 0] = 1.0\n        inv = 1.0 / counts\n        self.class_probs = inv / inv.sum()\n        self.indices_by_class = {c: np.where(y == c)[0] for c in range(n_classes)}\n\n    def __len__(self):\n        return math.ceil(len(self.y) / self.batch_size)\n\n    def __getitem__(self, idx):\n        chosen_classes = self.rng.choice(self.n_classes, size=self.batch_size, p=self.class_probs)\n        batch_idx = []\n        for c in chosen_classes:\n            candidates = self.indices_by_c

In [16]:
"""
def init_run(run_name='cnn_loso'):
    now = datetime.now()
    display_ts = now.strftime('%m/%d/%Y %I:%M:%S %p EST')
    folder_ts = now.strftime('%m-%d-%Y_%I-%M-%S_%p_EST').lower()
    run_id = f'{folder_ts}_{run_name}'
    run_dir = RUNS_DIR / run_id

    subdirs = {
        'models': run_dir / 'models',
        'metrics': run_dir / 'metrics',
        'plots': run_dir / 'plots',
        'artifacts': run_dir / 'artifacts',
    }
    for path in subdirs.values():
        path.mkdir(parents=True, exist_ok=True)

    manifest = {
        'run_id': run_id,
        'created_at': display_ts,
        'n_samples': int(len(y_all)),
        'input_shape': [int(X_cnn.shape[1]), int(X_cnn.shape[2])],
        'class_distribution': {int(c): int((y_all == c).sum()) for c in sorted(set(y_all.tolist()))},
    }
    (run_dir / 'run_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    (LATEST_DIR / 'latest_run.txt').write_text(str(run_dir), encoding='utf-8')

    return {
        'run_id': run_id,
        'run_dir': run_dir,
        'models_dir': subdirs['models'],
        'metrics_dir': subdirs['metrics'],
        'plots_dir': subdirs['plots'],
        'artifacts_dir': subdirs['artifacts'],
    }


def save_metrics(run_ctx, name, metrics_dict):
    path = run_ctx['metrics_dir'] / f'{name}.json'
    path.write_text(json.dumps(metrics_dict, indent=2), encoding='utf-8')
    return path


def next_plot_path(run_ctx, name, ext='png'):
    return run_ctx['plots_dir'] / f'{name}.{ext}'


RUN_CNN = init_run('cnn_loso_raw')
print(f"CNN run directory: {RUN_CNN['run_dir']}")
"""

'\ndef init_run(run_name=\'cnn_loso\'):\n    now = datetime.now()\n    display_ts = now.strftime(\'%m/%d/%Y %I:%M:%S %p EST\')\n    folder_ts = now.strftime(\'%m-%d-%Y_%I-%M-%S_%p_EST\').lower()\n    run_id = f\'{folder_ts}_{run_name}\'\n    run_dir = RUNS_DIR / run_id\n\n    subdirs = {\n        \'models\': run_dir / \'models\',\n        \'metrics\': run_dir / \'metrics\',\n        \'plots\': run_dir / \'plots\',\n        \'artifacts\': run_dir / \'artifacts\',\n    }\n    for path in subdirs.values():\n        path.mkdir(parents=True, exist_ok=True)\n\n    manifest = {\n        \'run_id\': run_id,\n        \'created_at\': display_ts,\n        \'n_samples\': int(len(y_all)),\n        \'input_shape\': [int(X_cnn.shape[1]), int(X_cnn.shape[2])],\n        \'class_distribution\': {int(c): int((y_all == c).sum()) for c in sorted(set(y_all.tolist()))},\n    }\n    (run_dir / \'run_manifest.json\').write_text(json.dumps(manifest, indent=2), encoding=\'utf-8\')\n    (LATEST_DIR / \'latest_run

In [ ]:
def build_cnn_model(input_shape=(3000, 1), n_classes=N_CLASSES):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.Conv1D(32, kernel_size=7, padding='same', activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling1D(pool_size=2),

        tf.keras.layers.Conv1D(64, kernel_size=5, padding='same', activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling1D(pool_size=2),

        tf.keras.layers.Conv1D(128, kernel_size=3, padding='same', activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.GlobalAveragePooling1D(),

        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.Dense(n_classes, activation='softmax')
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


model = build_cnn_model(input_shape=X_cnn.shape[1:])
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_6 (Conv1D)               │ (None, 3000, 32)       │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 3000, 32)       │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_4 (MaxPooling1D)  │ (None, 1500, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_7 (Conv1D)               │ (None, 1500, 64)       │        10,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 1500, 64)       │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_5 (MaxPooling1D)  │ (None, 750, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_8 (Conv1D)               │ (None, 750, 128)       │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 750, 128)       │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_2      │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 53,317 (208.27 KB)

 Trainable params: 52,869 (206.52 KB)

 Non-trainable params: 448 (1.75 KB)

In [18]:
def confusion_matrix_np(y_true, y_pred, n_classes=N_CLASSES):
    cm = np.zeros((n_classes, n_classes), dtype=np.int64)
    for t, p in zip(y_true, y_pred):
        cm[int(t), int(p)] += 1
    return cm


def cohen_kappa_np(cm):
    total = cm.sum()
    po = np.trace(cm) / total
    row_marginals = cm.sum(axis=1)
    col_marginals = cm.sum(axis=0)
    pe = np.sum(row_marginals * col_marginals) / (total ** 2)
    return float((po - pe) / (1 - pe + 1e-12))


def classification_metrics_np(y_true, y_pred, n_classes=N_CLASSES):
    cm = confusion_matrix_np(y_true, y_pred, n_classes=n_classes)
    support = cm.sum(axis=1)
    pred_count = cm.sum(axis=0)
    tp = np.diag(cm)

    precision = tp / np.maximum(pred_count, 1)
    recall = tp / np.maximum(support, 1)
    f1 = 2 * precision * recall / np.maximum(precision + recall, 1e-12)

    accuracy = float(tp.sum() / np.maximum(cm.sum(), 1))
    balanced_accuracy = float(np.mean(recall))
    macro_f1 = float(np.mean(f1))
    kappa = cohen_kappa_np(cm)

    return {
        'accuracy': accuracy,
        'balanced_accuracy': balanced_accuracy,
        'macro_f1': macro_f1,
        'cohen_kappa': kappa,
        'precision_per_class': precision.tolist(),
        'recall_per_class': recall.tolist(),
        'f1_per_class': f1.tolist(),
        'support_per_class': support.astype(int).tolist(),
        'confusion_matrix': cm.astype(int).tolist(),
    }


def plot_confusion_matrix(cm, title, save_path):
    fig, ax = plt.subplots(figsize=(6.2, 5.2))
    im = ax.imshow(cm, cmap='Blues')
    ax.set_title(title)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_xticks(range(N_CLASSES))
    ax.set_yticks(range(N_CLASSES))
    ax.set_xticklabels([STAGE_NAMES[i] for i in range(N_CLASSES)], rotation=30, ha='right')
    ax.set_yticklabels([STAGE_NAMES[i] for i in range(N_CLASSES)])

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, int(cm[i, j]), ha='center', va='center', fontsize=9)

    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    fig.savefig(save_path, dpi=140)
    plt.close(fig)


def plot_history(history, save_path):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    axes[0].plot(history.history['loss'], label='train')
    axes[0].plot(history.history['val_loss'], label='val')
    axes[0].set_title('Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].legend()

    axes[1].plot(history.history['accuracy'], label='train')
    axes[1].plot(history.history['val_accuracy'], label='val')
    axes[1].set_title('Accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].legend()

    fig.tight_layout()
    fig.savefig(save_path, dpi=140)
    plt.close(fig)


In [ ]:
from sklearn.model_selection import train_test_split

EPOCHS = 12
BATCH_SIZE = 32

# stratified random split for speed
X_train, X_temp, y_train, y_temp = train_test_split(X_cnn, y_all, test_size=0.3, random_state=42, stratify=y_all)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(f'Train: {len(y_train)} | Val: {len(y_val)} | Test: {len(y_test)}')

#class_counts = np.bincount(y_train, minlength=N_CLASSES)
#class_weight = {cls: float(len(y_train) / (N_CLASSES * max(class_counts[cls], 1))) for cls in range(N_CLASSES)}

model = build_cnn_model(input_shape=X_cnn.shape[1:])

class PerClassF1Callback(tf.keras.callbacks.Callback):
    def __init__(self, X_val, y_val, stage_names):
        super().__init__()
        self.X_val = X_val
        self.y_val = y_val
        self.stage_names = stage_names

    def on_epoch_end(self, epoch, logs=None):
        y_prob = self.model.predict(self.X_val, verbose=0)
        y_pred = np.argmax(y_prob, axis=1)
        epoch_metrics = classification_metrics_np(self.y_val, y_pred)
        f1s = epoch_metrics['f1_per_class']
        f1_text = ' | '.join(f'{self.stage_names[i]} F1: {f1s[i]:.3f}' for i in range(len(self.stage_names)))
        print(f'\nEpoch {epoch + 1} val per-class F1 | {f1_text}')


callbacks = [
    PerClassF1Callback(X_val, y_val, STAGE_NAMES),
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-5),
]


start = time.time()
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1,
)
elapsed = time.time() - start

y_prob = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_prob, axis=1)
metrics = classification_metrics_np(y_test, y_pred)
cm = np.array(metrics['confusion_matrix'])

run_record = {
    'n_train': int(len(y_train)),
    'n_val': int(len(y_val)),
    'n_test': int(len(y_test)),
    'epochs_ran': int(len(history.history['loss'])),
    'elapsed_sec': round(float(elapsed), 1),
    'accuracy': metrics['accuracy'],
    'balanced_accuracy': metrics['balanced_accuracy'],
    'macro_f1': metrics['macro_f1'],
    'cohen_kappa': metrics['cohen_kappa'],
    'per_stage_f1': {STAGE_NAMES[i]: float(metrics['f1_per_class'][i]) for i in range(N_CLASSES)},
    'confusion_matrix': metrics['confusion_matrix'],
}

from pathlib import Path
import json

# Make simple output folders
save_dir = Path("outputs") / "cnn_single_run"
models_dir = save_dir / "models"
plots_dir = save_dir / "plots"
metrics_dir = save_dir / "metrics"

models_dir.mkdir(parents=True, exist_ok=True)
plots_dir.mkdir(parents=True, exist_ok=True)
metrics_dir.mkdir(parents=True, exist_ok=True)

# Save artifacts
model_path = models_dir / "cnn_single_run.keras"
model.save(model_path)

hist_path = plots_dir / "cnn_history_single.png"
plot_history(history, hist_path)

cm_path = plots_dir / "cnn_confusion_matrix_single.png"
plot_confusion_matrix(cm, "CNN Single-Run Confusion Matrix", cm_path)

summary_path = metrics_dir / "cnn_single_run_summary.json"
summary_path.write_text(
    json.dumps({"run": run_record}, indent=2),
    encoding="utf-8"
)

print()
print(f"Elapsed: {elapsed:.1f}s | Epochs: {len(history.history['loss'])}")
print(f"Accuracy: {metrics['accuracy']*100:.2f}% | Macro F1: {metrics['macro_f1']*100:.2f}%")
print(f"Saved model: {model_path} | summary: {summary_path} | CM: {cm_path}")

Train: 76451 | Val: 16383 | Test: 16383
Epoch 1/12
2389/2390 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - accuracy: 0.8747 - loss: 0.3658
Epoch 1 val per-class F1 | Wake F1: 0.976 | N1 F1: 0.323 | N2 F1: 0.834 | N3 F1: 0.798 | REM F1: 0.703
2390/2390 ━━━━━━━━━━━━━━━━━━━━ 156s 64ms/step - accuracy: 0.8980 - loss: 0.2857 - val_accuracy: 0.9044 - val_loss: 0.2547 - learning_rate: 0.0010
Epoch 2/12
2389/2390 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - accuracy: 0.9168 - loss: 0.2250
Epoch 2 val per-class F1 | Wake F1: 0.982 | N1 F1: 0.277 | N2 F1: 0.812 | N3 F1: 0.757 | REM F1: 0.675
2390/2390 ━━━━━━━━━━━━━━━━━━━━ 150s 63ms/step - accuracy: 0.9183 - loss: 0.2202 - val_accuracy: 0.9057 - val_loss: 0.2426 - learning_rate: 0.0010
Epoch 3/12
2389/2390 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - accuracy: 0.9230 - loss: 0.2078
Epoch 3 val per-class F1 | Wake F1: 0.984 | N1 F1: 0.372 | N2 F1: 0.849 | N3 F1: 0.822 | REM F1: 0.690
2390/2390 ━━━━━━━━━━━━━━━━━━━━ 151s 63ms/step - accuracy: 0.9239 - loss: 0.2058 - val_accura

In [20]:
"model" in globals(), "history" in globals(), "cm" in globals()

(True, True, True)

In [21]:
from pathlib import Path
from datetime import datetime
import json

if "RUN_CNN" not in globals():
    RUNS_DIR = Path("outputs") / "runs"
    LATEST_DIR = Path("outputs")
    RUNS_DIR.mkdir(parents=True, exist_ok=True)
    LATEST_DIR.mkdir(parents=True, exist_ok=True)

    now = datetime.now()
    run_id = now.strftime("%m-%d-%Y_%I-%M-%S_%p_EST").lower() + "_cnn_single_recovered"
    run_dir = RUNS_DIR / run_id

    RUN_CNN = {
        "run_id": run_id,
        "run_dir": run_dir,
        "models_dir": run_dir / "models",
        "metrics_dir": run_dir / "metrics",
        "plots_dir": run_dir / "plots",
        "artifacts_dir": run_dir / "artifacts",
    }

    for p in ["models_dir", "metrics_dir", "plots_dir", "artifacts_dir"]:
        RUN_CNN[p].mkdir(parents=True, exist_ok=True)

    (run_dir / "run_manifest.json").write_text(
        json.dumps({"run_id": run_id, "created_at": now.isoformat()}, indent=2),
        encoding="utf-8",
    )
    (LATEST_DIR / "latest_run.txt").write_text(str(run_dir), encoding="utf-8")

if "save_metrics" not in globals():
    def save_metrics(run_ctx, name, metrics_dict):
        path = run_ctx["metrics_dir"] / f"{name}.json"
        path.write_text(json.dumps(metrics_dict, indent=2), encoding="utf-8")
        return path

if "next_plot_path" not in globals():
    def next_plot_path(run_ctx, name, ext="png"):
        return run_ctx["plots_dir"] / f"{name}.{ext}"

model_path = RUN_CNN["models_dir"] / "cnn_single_run.keras"
model.save(model_path)

hist_path = next_plot_path(RUN_CNN, "cnn_history_single")
plot_history(history, hist_path)

cm_path = next_plot_path(RUN_CNN, "cnn_confusion_matrix_single")
plot_confusion_matrix(cm, "CNN Single-Run Confusion Matrix", cm_path)

summary_path = save_metrics(RUN_CNN, "cnn_single_run_summary", {"run": run_record})

print(f"Saved model: {model_path}")
print(f"Saved summary: {summary_path}")
print(f"Saved history plot: {hist_path}")
print(f"Saved confusion matrix: {cm_path}")

Saved model: outputs\runs\04-29-2026_11-14-33_am_est_cnn_single_recovered\models\cnn_single_run.keras
Saved summary: outputs\runs\04-29-2026_11-14-33_am_est_cnn_single_recovered\metrics\cnn_single_run_summary.json
Saved history plot: outputs\runs\04-29-2026_11-14-33_am_est_cnn_single_recovered\plots\cnn_history_single.png
Saved confusion matrix: outputs\runs\04-29-2026_11-14-33_am_est_cnn_single_recovered\plots\cnn_confusion_matrix_single.png
